TODO:

[ ]: create training loop \
[ ]: resize inputs beforehand \
[ ]: use torchmetrics \
[ ]: integrate wandb \
[ ]: make it script (not ipynb) \
[ ]: integrate gradcam

In [1]:
import timm
from torchvision import transforms as pth_transforms
from torchvision import datasets
import torch
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torchmetrics.classification import MulticlassConfusionMatrix, Accuracy

assert torch.cuda.is_available(), "No GPU, no training!"
device = "cuda:0"

/home/stannis/.conda/envs/newTorch/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setting seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Some constants
NUM_CLASSES = 4
BATCH_SIZE = 32

# Some hyperparameters
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
SPLIT_RATIO = [0.7, 0.15, 0.15] # train, val, test (3rd one is inferred)


In [3]:
# Construct datasets, dataloaders

transform = pth_transforms.Compose([
    pth_transforms.RandomResizedCrop(224),
    pth_transforms.RandomHorizontalFlip(),
    pth_transforms.ToTensor(),
    pth_transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

my_dataset=datasets.ImageFolder(root='dl_action_dataset', transform=transform)

# split the dataset in train, validation and test set


train_size = int(SPLIT_RATIO[0] * len(my_dataset))
val_size = int(SPLIT_RATIO[1] * len(my_dataset))
test_size = len(my_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(my_dataset, [train_size, val_size, test_size])


train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
def freeze_all_except_head(model):
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False

In [ ]:
# Construct model, loss function, optimizer, scheduler
critereon = torch.nn.CrossEntropyLoss()
model = timm.create_model('vit_base_patch16_224', pretrained=True)
model.head = torch.nn.Linear(model.head.in_features, NUM_CLASSES)

freeze_all_except_head(model)

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [5]:
my_dataset.classes

['cut', 'pour', 'slide', 'stir']

In [ ]:
def run_one_epoch(model, loader, mode, criterion, optimizer, epoch):
    if mode == 'train':
        model.train()
    else:
        model.eval()


    acc = Accuracy(num_classes=NUM_CLASSES)
    conf_mat = MulticlassConfusionMatrix(num_classes=NUM_CLASSES)

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        with torch.set_grad_enabled(mode == 'train'):
            outputs = model(images)
            loss = criterion(outputs, labels)

            if mode == 'train':
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        acc.update(outputs.cpu(), labels.cpu())
        conf_mat.update(outputs.cpu(), labels.cpu())


    metrics = {
        'loss': loss.item(),
        'accuracy': acc.compute(),
        'confusion_matrix': conf_mat.compute()
    }

    return metrics

In [ ]:
# Pick N images from each class for visualization
N = 6
samples = {}

# Before resizing
h, w = 224, 224

for c in range(NUM_CLASSES):
    samples[c] = []
    for i in range(len(my_dataset)):
        if my_dataset[i][1] == c:
            samples[c].append(my_dataset[i][0])
        if len(samples[c]) == N:
            break

# Visualize the images
import matplotlib.pyplot as plt
import numpy as np
import torchvision

fig, axes = plt.subplots(NUM_CLASSES, N, figsize=(N, NUM_CLASSES))

for c in range(NUM_CLASSES):
    for i in range(N):
        axes[c][i].imshow(np.transpose(samples[c][i], (1, 2, 0)))
        axes[c][i].axis('off')

plt.show()